In [ ]:
# ==========================================
# # 1. SETUP DE AMBIENTE (Importações de bibliotecas)
# ==========================================

import pandas as pd  # Manipulação de tabelas e dados estruturados
import numpy as np # Operações matemáticas e vetoriais
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report # Métricas de Avalicação
from sklearn.model_selection import train_test_split # Divisão da base
from sklearn.preprocessing import StandardScaler # Normalização de escala
from tensorflow import keras # Framework de Deep Learning       # type: ignore
from tensorflow.keras import layers # Camadas da rede neural    # type: ignore

In [ ]:
# ==========================================
# # 2. CARGA E EXPLORAÇÃO (Leitura dos dados, .info(), gráficos)
# ==========================================

df = pd.read_csv('vendas.csv') # Lê a entrada
df.head() # Exibe as primeiras 5 linhas do arquivo de entrada
print(df.info()) # Informações sobre tipo de dados, linhas e memória
print(df.describe()) # Extrai e exibe estatísticas descritivas
print(df.isnull().sum()) # Mapeia dados nulos

In [ ]:
# ==========================================
# # 3. PRÉ-PROCESSAMENTO (Limpeza de nulos, conversão de textos, engenharia de atributos)
# ==========================================

df['idade'] = df['idade'].fillna(df['idade'].median()) # Substitui valores de idades nulos pela mediana
df['gasto_total'] = df['gasto_total'].fillna(0) # Substitui valores de gasto_mensal por 0 levando em consideração o tempo de contrato neste caso específico
df['meses_contrato'] = df['meses_contrato'].fillna(df['meses_contrato'].median()) # Substitui valores de meses_contrato pela mediana
# Cria uma nova coluna chamada gasto_mensal para ajudar a entender o comportamento do cliente
# Insere a nova coluna apenas se ela ainda não existir
if 'gasto_mensal' not in df.columns:
    df.insert(5, 'gasto_mensal', df['gasto_total'] / df['meses_contrato'])
df

In [ ]:
# ==========================================
# 4. PREPARAÇÃO PARA O MODELO (Separa X e y, divide treino e teste, padronização de dados númericos)
# ==========================================

# Separação de X preditores(sem ID,nome e classe) e y classe(isolado)
X = df.drop(columns=['id_cliente', 'nome', 'cancelou'])
y = df['cancelou']
y = y.map({'Nao': 0, 'Sim': 1}) ## Converte a classe categórica em numérica: 0 para Não e 1 para Sim
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) # Separa 80% para treino e 20% para teste. random_state garante que pode ser reproduzido de maneira semelhante
scaler = StandardScaler() # Padroniza os dados númericos
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# ==========================================
# # 5. TREINAMENTO DO MODELO
# ==========================================

# Inicializa um modelo Sequencial e guarda na varivel model
model = keras.Sequential([
    layers.Dense(32, activation='relu', input_shape=(X_train_scaled.shape[1],)), # Camada de Processamento Inicial(com 32 neurônios e ativação Relu introduz não-linearidade)e que também define o formato dos dados de entrada (input_shape) usando o número de colunas de X
    layers.Dropout(0.2), # Desativa aleatoriamente 20% dos neurônios para evitar overfitting
    layers.Dense(16, activation='relu'), # Segunda camada oculta com 16 neurônios e ativação relu para refinar os padrões aprendidos
    layers.Dense(1, activation='sigmoid') # Camada de saída com 1 neurônio e ativação Sigmoid para retornar uma probabilidade entre 0 e 1 (classificação binária)
])

# Compila o modelo definindo o algoritmo de otimização adam, a função de perda binária binary_crossentropy e a métrica de avaliação accuracy
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Treina a rede neural salvando o histórico na variável treino
# Passa os dados de treino padronizados (X_train_scaled e y_train)
# Executa 25 épocas (ciclos completos de leitura da base)
# Processa os dados em lotes (batch_size) de 32 em 32 linhas
# Separa 10% dos dados (validation_split) para validação em tempo real
# Mostra a barra de progresso do aprendizado na tela (verbose=1)
treino = model.fit(X_train_scaled, y_train, epochs=25, batch_size=32, validation_split=0.1, verbose=1)

In [ ]:
# =======================================================
# 6. AVALIAÇÃO DE PERFORMANCE
# =======================================================

# 1. Avalia o modelo diretamente com os dados de teste (Retorna a perda e a acurácia global)
loss, accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)

# 2. Gera as probabilidades de cancelamento (valores entre 0 e 1) para a base de teste
y_pred_probs = model.predict(X_test_scaled, verbose=0)

# 3. Converte as probabilidades em classes binárias: vira 1 se for maior que 0.5, senão vira 0
y_pred = (y_pred_probs > 0.5).astype(int)

# 4. Calcula as métricas de classificação cruzando as respostas reais (y_test) com as previsões (y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

# 5. Exibe o resumo de todas as métricas formatadas com 4 casas decimais
print("--- Resumo das Métricas de Performance ---")
print(f"Acurácia:  {accuracy:.4f} -> Porcentagem geral de acertos do modelo.")
print(f"Precision: {precision:.4f} -> Dos que o modelo previu que iam cancelar, quantos realmente cancelaram.")
print(f"Recall:    {recall:.4f} -> De todos os que realmente cancelaram, quantos o modelo conseguiu encontrar.")
print(f"F1-Score:  {f1:.4f} -> Equilíbrio (média harmônica) entre Precision e Recall.")

# 6. Gera e estrutura a Matriz de Confusão em um DataFrame para leitura clara
matriz = confusion_matrix(y_test, y_pred)
matriz_df = pd.DataFrame(
    matriz, 
    columns=['Previu: Ficou (0)', 'Previu: Cancelou (1)'],
    index=['Real: Ficou (0)', 'Real: Cancelou (1)']
)

print("\n--- Matriz de Confusão ---")
display(matriz_df) # Usa display() em vez de print para tabelas ficarem lindas no Jupyter/Colab
